In [16]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ─── PySpark ─────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import count, when, col


# ─── Data / ML ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, r2_score
)
from sklearn.preprocessing import label_binarize

# ─── Visualisation ────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})
PALETTE = sns.color_palette('tab10')

print('All imports OK')

All imports OK


In [17]:
# ─── Spark Session ────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName('NYC_HVFHV_Demand_Supply')

    # Core
    .master("local[*]")

    # Memory (quan trọng)
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")

    # Shuffle tuning (RẤT QUAN TRỌNG)
    .config("spark.sql.shuffle.partitions", "64")

    # Adaptive Query Execution
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")

    # Parquet handling
    .config("spark.sql.parquet.mergeSchema", "false")
    .config("spark.sql.parquet.filterPushdown", "true")

    # Timezone
    .config("spark.sql.session.timeZone", "America/New_York")

    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 3.5.7


In [18]:
BRONZE_PATH = 'datasets/fvhfv/*'

trip = (
    spark.read
    # .schema(schema)
    .parquet(BRONZE_PATH)
)
trip.count(), len(trip.columns)

(504000505, 24)

In [24]:
ok = StructType([
    StructField("time", TimestampNTZType(), True),
    StructField("temperature_2m", DoubleType(), True),
    StructField("rain", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
])
weather = spark.read.schema(ok).csv("datasets/newyork_weather_2024_2026.csv", header=True )

In [20]:
trips = trip.where("base_passenger_fare > 0 and driver_pay > 0 and trip_miles < 500 and trip_time < 20000 and driver_pay < 1500 ")
trips.count()

503325384

In [21]:
# ─── Window Configuration ─────────────────────────────────────────────────────
WINDOW_SECONDS = 180 * 60   # 30 min in seconds
SLIDE_SECONDS  =  60 * 60   # 10 min  in seconds

def epoch_to_window_end(ts_col: str, window_s: int = WINDOW_SECONDS, slide_s: int = SLIDE_SECONDS):
    """
    Assign each timestamp to the next (future) window boundary.
    window_end = ceil(ts / slide) * slide  — snaps to the next slide boundary.
    Spark's window() function handles this natively.
    """
    return F.window(F.col(ts_col), f'{window_s} seconds', f'{slide_s} seconds')

In [22]:
demand_all = (
    trips
    # .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("request_datetime"))
    .groupBy("window")
    .agg(
        # volume
        F.count("*").alias("request"),
    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("window_end <= '2026-01-01 05:00:00' and window_end >= '2024-01-01 05:00:00'")
demand_all = demand_all.sort("window_end")

In [27]:
import matplotlib.dates as mdates
demand_all_pd = demand_all.toPandas().set_index("window_end").sort_index()
weather_p = weather.toPandas().set_index("time").sort_index()  
demand_pd = demand_all_pd.diff(24).dropna()
weather_pd = weather_p.diff().diff(8760).dropna()
fig, ax = plt.subplots(3,1,figsize=(30, 15))
gold = pd.merge(
    demand_pd[["request"]],
    weather_pd,
    left_index=True, right_index=True, how="inner"
).dropna()
df = pd.merge(
    demand_all_pd[["request"]],
    weather_p,
    left_index=True, right_index=True, how="inner"
).dropna()

# Vẽ data
ax[0].plot(
    df[df.index < '2024-07-01'].index,
    df[df.index < '2024-07-01']['request'],
    alpha=0.7, label="request"
)
ax[1].plot(
    df.index,
    df["precipitation"],
    color = "#2F01E96B",
    alpha=0.7, label="precipitation"
)
ax[2].plot(
    df.index,
    df['rain'],
    color = "#0439FB",
    alpha=0.7, label="rain"
)
# Major tick: mỗi tháng
ax[0].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax[0].xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax[1].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax[2].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax[2].xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))

# Minor tick: mỗi ngày
ax[0].xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))
ax[1].xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))
ax[2].xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))

ax[0].grid(axis="x", which="major", linestyle="-",  linewidth=0.8, alpha=0.6)
ax[0].grid(axis="x", which="minor", linestyle=":",  linewidth=0.5, alpha=0.35)
ax[1].grid(axis="x", which="major", linestyle="-",  linewidth=0.8, alpha=0.6)
ax[1].grid(axis="x", which="minor", linestyle=":",  linewidth=0.5, alpha=0.35)
ax[2].grid(axis="x", which="major", linestyle="-",  linewidth=0.8, alpha=0.6)
ax[2].grid(axis="x", which="minor", linestyle=":",  linewidth=0.5, alpha=0.35)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller, acf, pacf, ccf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox
def adf_report(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"\n{'='*55}")
    print(f"ADF Test: {name}")
    print(f"{'='*55}")
    print(f"  ADF Statistic  : {result[0]:.4f}")
    print(f"  p-value        : {result[1]:.6f}")
    print(f"  Lags Used      : {result[2]}")
    print(f"  Critical (1%)  : {result[4]['1%']:.4f}")
    print(f"  Critical (5%)  : {result[4]['5%']:.4f}")
    print(f"  Critical (10%) : {result[4]['10%']:.4f}")
    if result[1] < 0.05:
        print(f"  ✅ STATIONARY (reject H0 at 5%)")
        return True, 0
    else:
        print(f"  ❌ NON-STATIONARY (fail to reject H0)")
        return False, 1
    
  
trip_stationary, d_trip = adf_report(gold.request, "trip_count")
temp_stationary, d_temp = adf_report(gold.temperature_2m, "temperature")



ADF Test: trip_count
  ADF Statistic  : -14.0121
  p-value        : 0.000000
  Lags Used      : 37
  Critical (1%)  : -3.4311
  Critical (5%)  : -2.8619
  Critical (10%) : -2.5669
  ✅ STATIONARY (reject H0 at 5%)

ADF Test: temperature
  ADF Statistic  : -18.2281
  p-value        : 0.000000
  Lags Used      : 37
  Critical (1%)  : -3.4311
  Critical (5%)  : -2.8619
  Critical (10%) : -2.5669
  ✅ STATIONARY (reject H0 at 5%)


In [ ]:
MAX_LAGS = 170   # enough to cover weekly lag 168
acf_vals  = acf(weather_p.temperature_2m.diff().dropna(),  nlags=MAX_LAGS, fft=True)
pacf_vals = pacf(weather_p.temperature_2m.diff().dropna(), nlags=MAX_LAGS, method="ywm")
 
conf_band = 1.96 / np.sqrt(len(weather_p))
 
fig, axes = plt.subplots(2, 1, figsize=(18, 15))
 
# ACF plot
axes[0].bar(range(len(acf_vals)), acf_vals, color="steelblue", width=0.5, label="ACF")
axes[0].axhline(conf_band,  color="red", linestyle="--", lw=0.8)
axes[0].axhline(-conf_band, color="red", linestyle="--", lw=0.8)
axes[0].axhline(0, color="black", lw=0.5)
axes[0].set_title("ACF – temperature_2m (post-differencing if applied)")
axes[0].set_xlim([0, MAX_LAGS])
for lag in [24, 168]:
    axes[0].axvline(lag, color="green", linestyle=":", lw=1.2, label=f"Lag {lag}")
axes[0].legend()
 
# PACF plot
axes[1].bar(range(len(pacf_vals)), pacf_vals, color="coral", width=0.5, label="PACF")
axes[1].axhline(conf_band,  color="red", linestyle="--", lw=0.8)
axes[1].axhline(-conf_band, color="red", linestyle="--", lw=0.8)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("PACF – temperature_2m")
axes[1].set_xlim([0, MAX_LAGS])
plt.tight_layout()
plt.show()


In [ ]:


MAX_LAGS = 170   # enough to cover weekly lag 168
acf_vals  = acf(gold.request.dropna(),  nlags=MAX_LAGS, fft=True)
pacf_vals = pacf(gold.request.dropna(), nlags=MAX_LAGS, method="ywm")
 
conf_band = 1.96 / np.sqrt(len(gold))
 
fig, axes = plt.subplots(2, 1, figsize=(18, 15))
 
# ACF plot
axes[0].bar(range(len(acf_vals)), acf_vals, color="steelblue", width=0.5, label="ACF")
axes[0].axhline(conf_band,  color="red", linestyle="--", lw=0.8)
axes[0].axhline(-conf_band, color="red", linestyle="--", lw=0.8)
axes[0].axhline(0, color="black", lw=0.5)
axes[0].set_title("ACF – temperature_2m (post-differencing if applied)")
axes[0].set_xlim([0, MAX_LAGS])
for lag in [24, 168]:
    axes[0].axvline(lag, color="green", linestyle=":", lw=1.2, label=f"Lag {lag}")
axes[0].legend()
 
# PACF plot
axes[1].bar(range(len(pacf_vals)), pacf_vals, color="coral", width=0.5, label="PACF")
axes[1].axhline(conf_band,  color="red", linestyle="--", lw=0.8)
axes[1].axhline(-conf_band, color="red", linestyle="--", lw=0.8)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("PACF – temperature_2m")
axes[1].set_xlim([0, MAX_LAGS])
plt.tight_layout()
plt.show()


In [ ]:
df = df.rename(columns={
    'request': 'request_count',
})
req_d24       = df["request_count"].diff(24)
temp_d8760    = df["temperature_2m"].diff(8760)
temp_d8760_d1 = temp_d8760.diff(1)
 
var_df = pd.DataFrame({
    "request":     req_d24,
    "temperature": temp_d8760_d1,
}).dropna()
 
# ── Align original request với var_df index ──
req_raw = df["request_count"].reindex(var_df.index)
 
# ── Train / Test split ──
HORIZON = 1
N       = len(var_df)
N_TRAIN = int(N * 0.8)
N_TEST  = N - N_TRAIN
 
var_train = var_df.iloc[:N_TRAIN]
var_test  = var_df.iloc[N_TRAIN:]
req_train = req_raw.iloc[:N_TRAIN]
req_test  = req_raw.iloc[N_TRAIN:]
 
print(f"Train : {N_TRAIN:,} hrs = {N_TRAIN/24:.0f} days")
print(f"Test  : {N_TEST:,}  hrs = {N_TEST/24:.0f}  days")
print(f"Horizon: t+{HORIZON} = {HORIZON} giờ ahead\n")

Train : 7,027 hrs = 293 days
Test  : 1,757  hrs = 73  days
Horizon: t+1 = 1 giờ ahead



In [ ]:
print("SARIMA(3,0,1)(1,1,1)[24]")
 
sarima_model = SARIMAX(
    req_train,
    order=(3, 0, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
    trend="c",
)
sarima_fit = sarima_model.fit(
    disp=False,
    maxiter=300,
    method="lbfgs",
)
print(sarima_fit.summary())
print(f"\nAIC = {sarima_fit.aic:.2f}")
print(f"BIC = {sarima_fit.bic:.2f}")

SARIMA(3,0,1)(1,1,1)[24]


c:\D\nam4_ki2\BigData\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\D\nam4_ki2\BigData\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


                                     SARIMAX Results                                      
Dep. Variable:                      request_count   No. Observations:                 7027
Model:             SARIMAX(3, 0, 1)x(1, 1, 1, 24)   Log Likelihood              -64862.679
Date:                            Sat, 04 Apr 2026   AIC                         129741.357
Time:                                    23:44:28   BIC                         129796.159
Sample:                                12-31-2024   HQIC                        129760.245
                                     - 10-20-2025                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      1.5446      1.765      0.875      0.382      -1.915       5.004
ar.L1          2.1743      0.018   

In [ ]:
sarima_fc    = sarima_fit.forecast(steps=N_TEST + HORIZON)
sarima_preds = sarima_fc.iloc[HORIZON: N_TEST + HORIZON].values
actual_sar   = req_test.values[:N_TEST]
 
print(f"\nSARIMA forecast done: {len(sarima_preds)} predictions")


SARIMA forecast done: 1757 predictions


In [ ]:
print("VAR — Lag Selection")
 
# Chọn maxlags phù hợp với RAM
# VAR(p) với 2 biến cần fit 2×2×p + 2 parameters
# Rule of thumb: N_TRAIN / (4p + 2) > 10
max_p_safe = int(N_TRAIN / 40)   # conservative
MAX_LAGS   = min(max_p_safe, 200)
print(f"N_TRAIN={N_TRAIN:,} → maxlags={MAX_LAGS} (safe upper bound)")
 
selector   = VAR(var_train)
lag_result = selector.select_order(maxlags=MAX_LAGS)
print(lag_result.summary())
 
p_aic = int(lag_result.aic)
p_bic = int(lag_result.bic)
p_var = p_aic
print(f"\nAIC optimal : p = {p_aic}")
print(f"BIC optimal : p = {p_bic}")
print(f"→ Dùng p = {p_var} (AIC ưu tiên forecast accuracy)")
 
if p_var >= 168:
    print("✅ p ≥ 168: VAR capture được weekly seasonality")
else:
    print(f"⚠️  p={p_var} < 168: weekly pattern sẽ còn trong residuals")
 
# ── Fit VAR ──
print(f"\nFitting VAR({p_var})...")
var_model = VAR(var_train)
var_fit   = var_model.fit(p_var)
print(var_fit.summary())
 
# ── Granger Causality Test ──
print("\n" + "-" * 50)
print("Granger Causality: temperature → request?")
print("-" * 50)
gc = var_fit.test_causality("request", "temperature", kind="f")
print(gc.summary())
gc_p = gc.pvalue
if gc_p < 0.05:
    print(f"✅ p={gc_p:.4f} < 0.05: temperature Granger-causes request")
    print("   → VAR justified: thời tiết có predictive value")
else:
    print(f"❌ p={gc_p:.4f} ≥ 0.05: temperature KHÔNG Granger-cause request")
    print("   → VAR ≈ univariate AR; weather không giúp predict request")
 
 
# ── VAR Walk-Forward Forecast + Invert diff(24) ──
print(f"\nVAR walk-forward forecast ({N_TEST} steps)...")
var_history   = var_train.values.tolist()
var_preds_d24 = []
 
for i in range(N_TEST):
    fc_in = np.array(var_history[-p_var:])
    fc    = var_fit.forecast(fc_in, steps=HORIZON)
    var_preds_d24.append(fc[-1, 0])          # request=col 0, t+HORIZON
    var_history.append(var_test.values[i])   # roll forward với actual
 
var_preds_d24 = np.array(var_preds_d24)
 
# Invert diff(24): y_t = y'_t + y_{t-24}
# var_preds_d24[i] → forecast của req_d24 tại index (N_TRAIN + i + HORIZON)
# cần y_{t-24} = req_raw tại (N_TRAIN + i + HORIZON - 24)
req_vals = req_raw.values
 
var_preds_orig = []
for i in range(N_TEST):
    idx_lag24 = N_TRAIN + i + HORIZON - 24
    if idx_lag24 >= 0:
        var_preds_orig.append(var_preds_d24[i] + req_vals[idx_lag24])
 
var_preds_orig = np.array(var_preds_orig)
n_valid      = len(var_preds_orig)
actual_var   = req_vals[N_TRAIN + HORIZON: N_TRAIN + HORIZON + n_valid]
 
print(f"VAR forecast done: {n_valid} predictions (original scale)")

VAR — Lag Selection
N_TRAIN=7,027 → maxlags=175 (safe upper bound)


c:\D\nam4_ki2\BigData\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


  VAR Order Selection (* highlights the minimums)  
        AIC         BIC         FPE         HQIC   
---------------------------------------------------
0         19.51       19.51   2.966e+08       19.51
1         17.28       17.28   3.184e+07       17.28
2         16.14       16.15   1.026e+07       16.15
3         15.90       15.91   8.011e+06       15.90
4         15.90       15.91   8.014e+06       15.90
5         15.81       15.83   7.323e+06       15.81
6         15.68       15.70   6.440e+06       15.69
7         15.68       15.71   6.422e+06       15.69
8         15.63       15.66   6.142e+06       15.64
9         15.61       15.65   6.042e+06       15.63
10        15.61       15.65   6.003e+06       15.62
11        15.60       15.64   5.947e+06       15.61
12        15.56       15.61   5.718e+06       15.58
13        15.56       15.61   5.712e+06       15.58
14        15.54       15.60   5.633e+06       15.56
15        15.53       15.59   5.568e+06       15.55
16        15

c:\D\nam4_ki2\BigData\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sat, 04, Apr, 2026
Time:                     23:44:45
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                    15.3126
Nobs:                     6853.00    HQIC:                   14.8567
Log likelihood:          -68834.1    FPE:                2.22857e+06
AIC:                      14.6167    Det(Omega_mle):     2.01782e+06
--------------------------------------------------------------------
Results for equation request
                      coefficient       std. error           t-stat            prob
-----------------------------------------------------------------------------------
const                    6.672482        18.905980            0.353           0.724
L1.request               2.007483         0.012381          162.138           0.000
L1.temperature         -42.340491        21.32

In [ ]:
def residual_diagnostics(actual, predicted, model_name, save_path=None):
    # ALIGN TIME INDEX
    actual = pd.Series(actual)
    predicted = pd.Series(predicted)
    actual, predicted = actual.align(predicted, join="inner")

    resid = actual - predicted
    cb    = 1.96 / np.sqrt(len(resid))
 
    print(f"\n{'═'*55}")
    print(f"RESIDUALS: {model_name}")
    print(f"{'═'*55}")
    print(f"  Mean     : {resid.mean():+.1f}   (expect ≈ 0)")
    print(f"  Std      : {resid.std():.1f}")
    print(f"  Skewness : {pd.Series(resid).skew():.3f}   (|skew|<1 → acceptable)")
    print(f"  Kurtosis : {pd.Series(resid).kurtosis():.3f}  (>3 → heavy tails)")
 
    # ACF của residuals
    res_acf  = acf(resid, nlags=170, fft=True)
    sig_lags = [i for i, v in enumerate(res_acf[1:], 1) if abs(v) > cb]
    print(f"\n  Conf band : ±{cb:.4f}")
    if sig_lags:
        print(f"  ⚠️  Significant ACF lags: {sig_lags[:8]}")
        if any(l in [24, 48] for l in sig_lags):
            print("     → Spike tại lag 24/48: daily pattern chưa fully captured")
        if any(l > 160 for l in sig_lags):
            print("     → Spike gần lag 168: weekly pattern chưa captured")
    else:
        print("  ✅ Residual ACF flat → white noise")
 
    # Ljung-Box
    lb = acorr_ljungbox(resid, lags=[12, 24, 48, 168], return_df=True)
    print(f"\n  Ljung-Box Test (H0: no autocorrelation):")
    all_pass = True
    for lag_val, row in lb.iterrows():
        ok   = row["lb_pvalue"] > 0.05
        icon = "✅" if ok else "❌"
        print(f"    lag={lag_val:4d}  stat={row['lb_stat']:8.2f}  p={row['lb_pvalue']:.4f}  {icon}")
        if not ok:
            all_pass = False
    if all_pass:
        print("  → ✅ ALL PASS: residuals are white noise")
    else:
        print("  → ❌ SOME FAIL: remaining autocorrelation in residuals")
        print("     (Ghi nhận là limitation của model)")
 
    # 4-panel plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    fig.suptitle(f"{model_name} — Residual Diagnostics", fontsize=13, fontweight="bold")
 
    # Panel 1: Residuals over time
    axes[0, 0].plot(resid, lw=0.4, color="steelblue", alpha=0.8)
    axes[0, 0].axhline(0, color="red", lw=0.8)
    axes[0, 0].set_title("Residuals over time")
    axes[0, 0].set_xlabel("Steps"); axes[0, 0].set_ylabel("Residual")
 
    # Panel 2: Distribution
    axes[0, 1].hist(resid, bins=80, color="steelblue", edgecolor="k", linewidth=0.2)
    axes[0, 1].set_title("Residual distribution (expect bell curve)")
    axes[0, 1].set_xlabel("Residual"); axes[0, 1].set_ylabel("Count")
 
    # Panel 3: ACF of residuals
    axes[1, 0].bar(range(len(res_acf)), res_acf, width=0.5, color="steelblue", alpha=0.7)
    axes[1, 0].axhline( cb, color="red",   linestyle="--", lw=0.8, label=f"±{cb:.3f}")
    axes[1, 0].axhline(-cb, color="red",   linestyle="--", lw=0.8)
    axes[1, 0].axhline(0,   color="black", lw=0.3)
    for vl, lbl in [(24, "daily"), (168, "weekly")]:
        axes[1, 0].axvline(vl, color="green", linestyle=":", lw=1, label=lbl)
    axes[1, 0].set_title("ACF of residuals (expect flat)")
    axes[1, 0].set_xlabel("Lag"); axes[1, 0].legend(fontsize=8)
 
    # Panel 4: Residual vs Fitted
    n_plot = min(2000, len(predicted))
    axes[1, 1].scatter(predicted[:n_plot], resid[:n_plot], alpha=0.1, s=2, color="steelblue")
    axes[1, 1].axhline(0, color="red", lw=0.8)
    axes[1, 1].set_title("Residual vs Fitted (expect random cloud)")
    axes[1, 1].set_xlabel("Fitted"); axes[1, 1].set_ylabel("Residual")
 
    plt.tight_layout()
    fname = save_path or f"outputs/residuals_{model_name[:20].replace(' ','_')}.png"
    plt.savefig(fname, dpi=130)
    plt.show()
    print(f"  Saved: {fname}")
    return resid
 
 
res_sarima = residual_diagnostics(
    actual_sar, sarima_preds,
    "SARIMA(3,0,1)(1,1,1)[24]",
    save_path="outputs/residuals_sarima.png"
)
res_var = residual_diagnostics(
    actual_var, var_preds_orig,
    f"VAR({p_var})",
    save_path="outputs/residuals_var.png"
)


═══════════════════════════════════════════════════════
RESIDUALS: SARIMA(3,0,1)(1,1,1)[24]
═══════════════════════════════════════════════════════
  Mean     : +2606.0   (expect ≈ 0)
  Std      : 23155.0
  Skewness : 1.374   (|skew|<1 → acceptable)
  Kurtosis : 3.773  (>3 → heavy tails)

  Conf band : ±0.0468
  ⚠️  Significant ACF lags: [1, 2, 3, 4, 5, 6, 7, 8]
     → Spike tại lag 24/48: daily pattern chưa fully captured
     → Spike gần lag 168: weekly pattern chưa captured

  Ljung-Box Test (H0: no autocorrelation):
    lag=  12  stat= 3640.88  p=0.0000  ❌
    lag=  24  stat= 4938.40  p=0.0000  ❌
    lag=  48  stat= 6066.25  p=0.0000  ❌
    lag= 168  stat= 9401.81  p=0.0000  ❌
  → ❌ SOME FAIL: remaining autocorrelation in residuals
     (Ghi nhận là limitation của model)


  Saved: outputs/residuals_sarima.png

═══════════════════════════════════════════════════════
RESIDUALS: VAR(174)
═══════════════════════════════════════════════════════
  Mean     : +103.6   (expect ≈ 0)
  Std      : 8286.8
  Skewness : 0.208   (|skew|<1 → acceptable)
  Kurtosis : 6.597  (>3 → heavy tails)

  Conf band : ±0.0468
  ⚠️  Significant ACF lags: [1, 2, 3, 4, 5, 6, 7, 8]
     → Spike tại lag 24/48: daily pattern chưa fully captured
     → Spike gần lag 168: weekly pattern chưa captured

  Ljung-Box Test (H0: no autocorrelation):
    lag=  12  stat= 1452.61  p=0.0000  ❌
    lag=  24  stat= 1486.13  p=0.0000  ❌
    lag=  48  stat= 1783.99  p=0.0000  ❌
    lag= 168  stat= 3863.10  p=0.0000  ❌
  → ❌ SOME FAIL: remaining autocorrelation in residuals
     (Ghi nhận là limitation của model)


  Saved: outputs/residuals_var.png


In [ ]:
def evaluate(actual, predicted, name):
    # ALIGN TIME INDEX (QUAN TRỌNG)
    actual = pd.Series(actual)
    predicted = pd.Series(predicted)
    actual, predicted = actual.align(predicted, join="inner")

    n    = len(actual)
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    mae  = np.mean(np.abs(actual - predicted))
    
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100
    
    # Symmetric MAPE
    smape = np.mean(
        2 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + 1e-8)
    ) * 100

    return {
        "Model"  : name,
        "N_test" : n,
        "RMSE"   : rmse,
        "MAE"    : mae,
        "MAPE%"  : mape,
        "sMAPE%" : smape,
    }
 
print("\n" + "═" * 70)
print(f"FINAL EVALUATION — Forecast t+{HORIZON} (original scale: request_count)")
print("═" * 70)
 
r1 = evaluate(actual_sar, sarima_preds,   "SARIMA(3,0,1)(1,1,1)[24]")
r2 = evaluate(actual_var, var_preds_orig, f"VAR({p_var})")
 
results_df = pd.DataFrame([r1, r2])
print(results_df.to_string(index=False))
 
# So sánh tương đối
rmse_sar = r1["RMSE"]
rmse_var = r2["RMSE"]
improvement = (rmse_sar - rmse_var) / rmse_sar * 100
 
print(f"\n{'─'*50}")
if improvement > 0:
    print(f"✅ VAR tốt hơn SARIMA: -{improvement:.1f}% RMSE")
    winner = f"VAR({p_var})"
elif improvement < -2:
    print(f"✅ SARIMA tốt hơn VAR: +{-improvement:.1f}% RMSE")
    winner = "SARIMA(3,0,1)(1,1,1)[24]"
else:
    print(f"≈  Hai model tương đương (diff={abs(improvement):.1f}%)")
    winner = "SARIMA (simpler → preferred)"
 
print(f"Winner: {winner}")
 
if gc_p >= 0.05:
    print("\n📌 Granger test: temperature KHÔNG significant")
    print("   → VAR improvement (nếu có) đến từ lag structure, không phải weather")
    print("   → Có thể dùng univariate SARIMA là đủ trong production")
 
# Save results
results_df.to_csv("outputs/model_comparison.csv", index=False)
print("\nSaved: outputs/model_comparison.csv")


══════════════════════════════════════════════════════════════════════
FINAL EVALUATION — Forecast t+1 (original scale: request_count)
══════════════════════════════════════════════════════════════════════
                   Model  N_test         RMSE          MAE     MAPE%    sMAPE%
SARIMA(3,0,1)(1,1,1)[24]    1757 23294.595990 16379.233023 22.676153 21.469858
                VAR(174)    1756  8285.107923  5178.857170  7.833318  8.354077

──────────────────────────────────────────────────
✅ VAR tốt hơn SARIMA: -64.4% RMSE
Winner: VAR(174)

📌 Granger test: temperature KHÔNG significant
   → VAR improvement (nếu có) đến từ lag structure, không phải weather
   → Có thể dùng univariate SARIMA là đủ trong production

Saved: outputs/model_comparison.csv


In [ ]:
plot_n = min(336, min(N_TEST, n_valid))   # 2 tuần = 14×24
time_idx = np.arange(plot_n)
 
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)
fig.suptitle(
    f"Forecast t+{HORIZON} vs Actual — First {plot_n//24} days of test set",
    fontsize=13, fontweight="bold"
)
 
# ── Panel 1: SARIMA ──
axes[0].plot(time_idx, actual_sar[:plot_n],   label="Actual",  color="black",      lw=1.2)
axes[0].plot(time_idx, sarima_preds[:plot_n], label="SARIMA",  color="royalblue",  lw=0.9, alpha=0.85)
axes[0].fill_between(
    time_idx,
    sarima_preds[:plot_n] - res_sarima[:plot_n].std(),
    sarima_preds[:plot_n] + res_sarima[:plot_n].std(),
    alpha=0.12, color="royalblue", label="±1σ"
)
axes[0].set_title(f"SARIMA(3,0,1)(1,1,1)[24]  |  RMSE={rmse_sar:,.0f}")
axes[0].set_ylabel("request_count"); axes[0].legend(loc="upper right")
 
# ── Panel 2: VAR ──
axes[1].plot(time_idx, actual_var[:plot_n],       label="Actual",         color="black",  lw=1.2)
axes[1].plot(time_idx, var_preds_orig[:plot_n],   label=f"VAR({p_var})",  color="tomato", lw=0.9, alpha=0.85)
axes[1].fill_between(
    time_idx,
    var_preds_orig[:plot_n] - res_var[:plot_n].std(),
    var_preds_orig[:plot_n] + res_var[:plot_n].std(),
    alpha=0.12, color="tomato", label="±1σ"
)
axes[1].set_title(f"VAR({p_var})  |  RMSE={rmse_var:,.0f}")
axes[1].set_ylabel("request_count"); axes[1].legend(loc="upper right")
 
# ── Panel 3: Error comparison ──
err_sar = np.abs(actual_sar[:plot_n]  - sarima_preds[:plot_n])
err_var = np.abs(actual_var[:plot_n]  - var_preds_orig[:plot_n])
axes[2].plot(time_idx, err_sar, label=f"|Error| SARIMA", color="royalblue", lw=0.7, alpha=0.8)
axes[2].plot(time_idx, err_var, label=f"|Error| VAR",    color="tomato",    lw=0.7, alpha=0.8)
axes[2].axhline(err_sar.mean(), color="royalblue", linestyle="--", lw=0.8, label=f"MAE SARIMA={err_sar.mean():.0f}")
axes[2].axhline(err_var.mean(), color="tomato",    linestyle="--", lw=0.8, label=f"MAE VAR={err_var.mean():.0f}")
axes[2].set_title("Absolute error |actual - predicted|")
axes[2].set_xlabel("Hours in test period")
axes[2].set_ylabel("|Error|"); axes[2].legend(loc="upper right")
 
plt.tight_layout()
plt.savefig("outputs/forecast_comparison.png", dpi=130)
plt.show()
print("Saved: outputs/forecast_comparison.png")

Saved: outputs/forecast_comparison.png


In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════╗
║                    PIPELINE SUMMARY                         ║
╠══════════════════════════════════════════════════════════════╣
║  Data                                                       ║
║    Train: {N_TRAIN:>6,} hrs ({N_TRAIN/24:>5.0f} days)                       ║
║    Test : {N_TEST:>6,} hrs ({N_TEST/24:>5.0f}  days)                       ║
║    Horizon: t+{HORIZON} (3 giờ ahead)                          ║
╠══════════════════════════════════════════════════════════════╣
║  SARIMA(3,0,1)(1,1,1)[24]                                   ║
║    RMSE : {rmse_sar:>12,.1f}                                   ║
║    MAE  : {r1['MAE']:>12,.1f}                                   ║
║    MAPE : {r1['MAPE%']:>11.2f}%                                  ║
║    sMAPE: {r1['sMAPE%']:>11.2f}%                                  ║
╠══════════════════════════════════════════════════════════════╣
║  VAR({p_var})                                                  ║
║    RMSE : {rmse_var:>12,.1f}                                   ║
║    MAE  : {r2['MAE']:>12,.1f}                                   ║
║    MAPE : {r2['MAPE%']:>11.2f}%                                  ║
║    sMAPE: {r2['sMAPE%']:>11.2f}%                                  ║
╠══════════════════════════════════════════════════════════════╣
║  Granger (temp → request): p={gc_p:.4f}                    ║
║  Winner: {winner:<46} ║
╚══════════════════════════════════════════════════════════════╝
""")
 


╔══════════════════════════════════════════════════════════════╗
║                    PIPELINE SUMMARY                         ║
╠══════════════════════════════════════════════════════════════╣
║  Data                                                       ║
║    Train:  7,027 hrs (  293 days)                       ║
║    Test :  1,757 hrs (   73  days)                       ║
║    Horizon: t+1 (3 giờ ahead)                          ║
╠══════════════════════════════════════════════════════════════╣
║  SARIMA(3,0,1)(1,1,1)[24]                                   ║
║    RMSE :     23,294.6                                   ║
║    MAE  :     16,379.2                                   ║
║    MAPE :       22.68%                                  ║
║    sMAPE:       21.47%                                  ║
╠══════════════════════════════════════════════════════════════╣
║  VAR(174)                                                  ║
║    RMSE :      8,285.1                                   ║
║